# PHREEQC PROJECT CODES

These notebook's function is to aide in comparing each REE's elemental values, comparing different k values in REE speciations, checking which Ree speciations are included in the database and which else should be included.

Imports

In [ ]:
import warnings
import re
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import build_database.clean_tables as ct
import build_database.utils as ut
import utils
import importlib.resources as pkg_resources
import build_database.databases
import pytest

Setup

In [2]:
# Enter the directory of your databases in the parenthesis below:
database_list = pkg_resources.files('build_database.databases')
database_list = ut.phreeqc_database_list(database_list)

# create Solution Species table
solution_species = ct.compile_solution_species_table(database_list)

# create Solution Master Species table
sms = ct.compile_master_solution_table(database_list, analysis=True)

# create Phases table
phases = ct.compile_phase_table(database_list)

# drop duplicate rows
before = len(sms)
filter_columns = ['element']
sms = sms.drop_duplicates(subset=filter_columns)
after = len(sms)
print(f'Filtered {before - after} duplicate rows from Solution Master Species table')

filter_columns = ['equation']
before = len(solution_species)
filter_columns = solution_species.columns[:-1]
solution_species = solution_species.drop_duplicates(subset=filter_columns)
after = len(solution_species)
print(f'Filtered {before - after} duplicate rows from Solution Species table')

filter_columns = ['phase_name']
before = len(phases)
filter_columns = phases.columns[:-1]
phases = phases.drop_duplicates(subset=filter_columns)
after = len(phases)
print(f'Filtered {before - after} duplicate rows from Phases table')

Filtered 399 duplicate rows from Solution Master Species table
Filtered 347 duplicate rows from Solution Species table
Filtered 0 duplicate rows from Phases table


In [ ]:
# Here is a list of all your database combined:
display(sms)
display(solution_species)
display(phases)

# If you want them exported into an excel file, simply activate the code below - and write your intended file name in the brackets:
with pd.ExcelWriter('database_lists.xlsx') as writer:
    sms.to_excel(writer, sheet_name='sms', index=False)
    solution_species.to_excel(writer, sheet_name='solution_species', index=False)
    phases.to_excel(writer, sheet_name = 'phases', index=False)


,element,species,alk,gfw_formula,element_gfw,source
0,Acetate,HAcetate,0.0,Acetate,59.,#llnl.dat
1,Ag,Ag+,0.0,Ag,107.8682,#llnl.dat
2,Ag(+1),Ag+,0.0,Ag,None,#llnl.dat
3,Ag(+2),Ag+2,0.0,Ag,None,#llnl.dat
4,Al,Al+3,0.0,Al,26.9815,#llnl.dat
...,...,...,...,...,...,...
103,Scn,Scn-,0.0,Scn,58.084,#sit.dat
114,Suberate,Suberate-2,0.0,Suberate,170.16,#sit.dat
115,Succinat,Succinat-2,0.0,Succinat,116.07,#sit.dat
24,Fulvate,Fulvate-2,0.0,650.,650.,#Tipping_Hurley.dat


,equation,log_k,delta_h,gamma,d_w,v_m,millero,activity_water,add_logk,llnl_gamma,co2_llnl_gamma,erm_ddl,no_check,mole_balance,source
12,Ce+3 = Ce+3,0.00,"(0, kJ/mol)",None,None,None,None,None,None,9.0,None,None,False,None,llnl.dat
18,Dy+3 = Dy+3,0.00,"(0, kJ/mol)",None,None,None,None,None,None,5.0,None,None,False,None,llnl.dat
20,Er+3 = Er+3,0.00,"(0, kJ/mol)",None,None,None,None,None,None,5.0,None,None,False,None,llnl.dat
22,Eu+3 = Eu+3,0.00,"(0, kJ/mol)",None,None,None,None,None,None,5.0,None,None,False,None,llnl.dat
26,Gd+3 = Gd+3,0.00,"(0, kJ/mol)",None,None,None,None,None,None,5.0,None,None,False,None,llnl.dat
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4098,1Sm+3 + 1Br- = SmBr+2,0.23,"(17.023,)",None,None,None,None,None,None,NaN,None,None,False,None,sit.dat
4099,1Sm+3 + 1Cl- = SmCl+2,0.72,"(22.277,)",None,None,None,None,None,None,NaN,None,None,False,None,sit.dat
4100,1Sm+3 + 1F- = SmF+2,4.21,"(24.180,)",None,None,None,None,None,None,NaN,None,None,False,None,sit.dat
4101,1Sm+3 + 2F- = SmF2+,6.43,"(18.850,)",None,None,None,None,None,None,NaN,None,None,False,None,sit.dat


,phase_name,dissolution_reaction,log_k,delta_h,analytic,v_m,t_c,p_c,omega,source
0,(UO2)2As2O7,(UO2)2As2O7 +2H+ +H2O = + 2H2AsO4- + 2UO2+2,7.7066,"(-145.281, kJ/mol)","(-1.6147e+002, -6.3487e-002, 1.0052e+004, 6.23...",None,None,None,None,llnl.dat
1,(UO2)2Cl3,(UO2)2Cl3 = + UO2+ + UO2+2 + 3Cl-,12.7339,"(-140.866, kJ/mol)","(-2.3895e+002, -9.2925e-002, 1.1722e+004, 9.69...",None,None,None,None,llnl.dat
2,(UO2)2P2O7,(UO2)2P2O7 +H2O = + 2HPO4-2 + 2UO2+2,-14.6827,"(-103.726, kJ/mol)","(-3.4581e+002, -1.3987e-001, 1.0703e+004, 1.36...",None,None,None,None,llnl.dat
3,(UO2)3(AsO4)2,(UO2)3(AsO4)2 +4H+ = + 2H2AsO4- + 3UO2+2,9.3177,"(-186.72, kJ/mol)","(-1.9693e+002, -7.3236e-002, 1.2936e+004, 7.46...",None,None,None,None,llnl.dat
4,(UO2)3(PO4)2,(UO2)3(PO4)2 +2H+ = + 2HPO4-2 + 3UO2+2,-14.0241,"(-149.864, kJ/mol)","(-3.6664e+002, -1.4347e-001, 1.3486e+004, 1.41...",None,None,None,None,llnl.dat
...,...,...,...,...,...,...,...,...,...,...
2741,HCl(g),HCl = 1H+ + 1Cl-,6.29,"(-74.770,)","(-6.80912E+0, 0E+0, 3.9055E+3, 0E+0, 0E+0)",None,None,None,None,sit.dat
2742,O2(g),O2 = 1O2,-2.9,"(-12.134,)","(-5.02578E+0, 0E+0, 6.33802E+2, 0E+0, 0E+0)",None,None,None,None,sit.dat
2743,SO2(g),SO2 = 2H+ + 1SO3-2 - 1H2O,-8.94,"(-48.420,)","(-1.74228E+1, 0E+0, 2.52915E+3, 0E+0, 0E+0)",None,None,None,None,sit.dat
2744,H2O(g),Ca(UO2)2(SiO3OH)2 + 6H+ = Ca+2 + 2UO2+2 + 2H4SiO4,17.489,"(-11.0, kcal)","(-0.26, 0.0, -731.0, 0.0, 0.0)",None,None,None,None,Tipping_Hurley.dat


In [23]:
# functions for searching desired values - use whichever you like to see the database values of certain equation/species/phases

# this is for the solution_master_speices block
def find_species(species:str):
    find_sp = sms[sms['species'].str.contains(species, regex=False)]
    return find_sp

# this is for the solution species
def find_equation(equation:str):
    equation = equation.replace(" ","")
    equation = re.sub(r'\b1(?!\d)', '', equation)
    find_eq = solution_species[solution_species['equation'].str.contains(equation, regex=False)]
    return find_eq

# this is for the phases block
def find_phases(species:str):
    find_phases = phases[phases['phase_name'].str.contains(species, regex = False)]
    return find_phases

# here are the example template. Type in the equation/find_species/find_phases in the brackets of each function:
display(find_species('La'))
display(find_equation('YbF'))
display(find_phases('Quartz'))

# If you want to export these, you can use codes below after altering it for your desired function/search value:
find_species('La').to_excel('La_species.xlsx', index = False)

,element,species,alk,gfw_formula,element_gfw,source
104,La,La+3,0.0,La,138.9055,#llnl.dat
105,La(+2),La+2,0.0,La,None,#llnl.dat
106,La(+3),La+3,0.0,La,None,#llnl.dat


,equation,log_k,delta_h,gamma,d_w,v_m,millero,activity_water,add_logk,llnl_gamma,co2_llnl_gamma,erm_ddl,no_check,mole_balance,source
1260,Yb+3 + F- = YbF+2,4.8085,"(23.2212, kJ/mol)",None,None,None,None,None,None,4.5,None,None,False,None,llnl.dat
1261,2F- + Yb+3 = YbF2+,8.3709,"(12.1336, kJ/mol)",None,None,None,None,None,None,4.0,None,None,False,None,llnl.dat
1262,3F- + Yb+3 = YbF3,11.0537,"(-13.1796, kJ/mol)",None,None,None,None,None,None,3.0,None,None,False,None,llnl.dat
1263,4F- + Yb+3 = YbF4-,13.2234,"(-60.2496, kJ/mol)",None,None,None,None,None,None,4.0,None,None,False,None,llnl.dat


,phase_name,dissolution_reaction,log_k,delta_h,analytic,v_m,t_c,p_c,omega,source
722,Quartz,SiO2 = + SiO2,-3.9993,"(32.949, kJ/mol)","(7.7698e-002, 1.0612e-002, 3.4651e+003, -4.355...",None,None,None,None,llnl.dat
1321,Quartz,SiO2 + 2H2O = H4SiO4,-4.0,"(22.36, kJ)",None,None,None,None,None,minteq.v4.dat
1790,Quartz,SiO2 + 2 H2O = H4SiO4,-3.98,"(5.990, kcal)","(0.41, 0.0, -1309.0)","(22.67,)",None,None,None,phreeqc.dat
2530,Quartz,SiO2 = 1H4(SiO4) - 2H2O,-3.74,"(21.166,)","(-3.18814E-2, 0E+0, -1.10558E+3, 0E+0, 0E+0)",None,None,None,None,sit.dat
